In [1]:
import pandas as pd
import numpy as np
from ast import literal_eval
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 读取数据
df = pd.read_csv('hardness_curve_dataset.csv')

# 2. 解析 hv_data 为数值数组
df['hv_data'] = df['hv_data'].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)
hv_array = np.array(df['hv_data'].tolist())  # (样本数, 序列长度)

# 3. 提取输入特征（排除'Image_Name' 和 'hv_data' 列）
feature_columns = ['C', 'Si', 'Mn', 'P', 'Cr', 'Perimeter', 'Aspect_Ratio',
                   'Compactness', 'Density', 'Uniformity', 'carburizing_temp', 'temp_gradient']
X = df[feature_columns].values
Y = hv_array

# 4. 划分训练和测试集
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# 5. 特征标准化
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("x_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)


X_train shape: (117, 12)
y_train shape: (117, 14)
x_test shape: (30, 12)
y_test shape: (30, 14)


In [2]:
# 转换数据类型以便后续使用PyTorch（深度学习部分）或保持NumPy用于传统ML部分
X_train_np = X_train.astype(np.float32)
X_test_np = X_test.astype(np.float32)
y_train_np = y_train.astype(np.float32)
y_test_np = y_test.astype(np.float32)

## 1 随机森林回归模型

In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import math

# 1. 初始化随机森林回归模型 (例如100棵树，可根据需要调整)
rf_model = RandomForestRegressor(n_estimators=100, random_state=0)
# 2. 在训练集上训练模型
rf_model.fit(X_train_np, y_train_np)

# 3. 在测试集上预测硬度曲线
y_pred_rf = rf_model.predict(X_test_np)

# 4. 计算评估指标
mse_rf = mean_squared_error(y_test_np, y_pred_rf)               # 均方误差
rmse_rf = math.sqrt(mse_rf)                                     # 均方根误差
r2_rf = r2_score(y_test_np, y_pred_rf)                          # R^2 得分

print(f"随机森林回归 - 测试集 MSE: {mse_rf:.3f}, RMSE: {rmse_rf:.3f}, R²: {r2_rf:.3f}")


随机森林回归 - 测试集 MSE: 170.650, RMSE: 13.063, R²: 0.861


## 2 支持向量回归模型 (SVR)

In [4]:
from sklearn.svm import SVR
from sklearn.multioutput import MultiOutputRegressor

# 1. 初始化支持向量回归模型，并用 MultiOutputRegressor 包装以支持多输出
base_svr = SVR(kernel='rbf', C=10.0, epsilon=0.01)
svr_model = MultiOutputRegressor(base_svr)
# 2. 训练SVR模型
svr_model.fit(X_train_np, y_train_np)

# 3. 在测试集上进行预测
y_pred_svr = svr_model.predict(X_test_np)

# 4. 计算评估指标
mse_svr = mean_squared_error(y_test_np, y_pred_svr)
rmse_svr = math.sqrt(mse_svr)
r2_svr = r2_score(y_test_np, y_pred_svr)

print(f"支持向量回归 - 测试集 MSE: {mse_svr:.3f}, RMSE: {rmse_svr:.3f}, R²: {r2_svr:.3f}")


支持向量回归 - 测试集 MSE: 294.793, RMSE: 17.170, R²: 0.844


## 3 XGBoost

In [5]:
from xgboost import XGBRegressor


# 6. 使用 XGBoost 进行多输出回归
# 初始化 XGBoost 回归模型
xgb_model = XGBRegressor(
    n_estimators=100,  # 树的数量
    learning_rate=0.1,  # 学习率
    max_depth=5,  # 树的最大深度
    random_state=42
)

# 使用 MultiOutputRegressor 包装 XGBoost 以支持多输出
multioutput_xgb = MultiOutputRegressor(xgb_model)

# 在训练集上训练模型
multioutput_xgb.fit(X_train_np, y_train_np)

# 在测试集上预测硬度曲线
y_pred_xgb = multioutput_xgb.predict(X_test_np)

# 7. 计算评估指标
mse_xgb = mean_squared_error(y_test_np, y_pred_xgb)  # 均方误差
rmse_xgb = math.sqrt(mse_xgb)  # 均方根误差
r2_xgb = r2_score(y_test_np, y_pred_xgb)  # R² 得分

print(f"XGBoost 回归 - 测试集 MSE: {mse_xgb:.3f}, RMSE: {rmse_xgb:.3f}, R²: {r2_xgb:.3f}")

XGBoost 回归 - 测试集 MSE: 131.368, RMSE: 11.462, R²: 0.905
